# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Hamna0696/flyrank-ml-internship/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

My rule: I will prioritize content that already has meaningful visibility but is becoming stale or losing performance. The score will reward items with higher impressions and older updates, while reason codes will explain why each item was selected.

Reason codes: stale_and_visible, high_visibility, needs_review.

In [2]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import pandas as pd
import numpy as np
from pathlib import Path

DATA_PATH = Path("content_refresh_anonymized.csv")
OUTPUT_PATH = Path("baseline_action_score.csv")

df = pd.read_csv(DATA_PATH)

print("Shape:", df.shape)
print("Columns:")
print(df.columns.tolist())

Shape: (30000, 44)
Columns:
['content_id', 'client_id', 'search_volume', 'competition', 'competition_level', 'cpc', 'content_type', 'main_intent', 'word_count', 'char_count', 'provider_used', 'model_used', 'impressions_90d', 'clicks_90d', 'pageviews_90d', 'sessions_90d', 'users_90d', 'engaged_sessions_90d', 'ai_sessions_90d', 'scroll_events_90d', 'days_with_impressions', 'days_with_sessions', 'impressions_last_30d', 'clicks_last_30d', 'sessions_last_30d', 'impressions_prev_30d', 'clicks_prev_30d', 'sessions_prev_30d', 'content_age_days', 'age_tier', 'age_tier_order', 'days_since_last_update', 'freshness_tier', 'word_count_tier', 'char_count_tier', 'ctr', 'avg_position', 'engagement_rate', 'scroll_rate', 'ai_traffic_pct', 'impression_tier', 'position_tier', 'trend_direction', 'trend_pct']


## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [8]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import pandas as pd
import numpy as np
from pathlib import Path

# Load dataset
df = pd.read_csv("content_refresh_anonymized.csv")

# --------------------------------------------------
# 1. Select the two baseline signals
# --------------------------------------------------

df["age_signal"] = pd.to_numeric(
    df["content_age_days"], errors="coerce"
)

df["visibility_signal"] = pd.to_numeric(
    df["impressions_90d"], errors="coerce"
)

# Fill missing values
df["age_signal"] = df["age_signal"].fillna(
    df["age_signal"].median()
)

df["visibility_signal"] = df["visibility_signal"].fillna(
    df["visibility_signal"].median()
)

# --------------------------------------------------
# 2. Convert signals to percentile scores
# --------------------------------------------------

# Older content = higher refresh priority
df["age_score"] = df["age_signal"].rank(pct=True)

# More impressions = higher business visibility
df["visibility_score"] = df["visibility_signal"].rank(pct=True)

# --------------------------------------------------
# 3. Calculate baseline action score
# --------------------------------------------------

df["action_score"] = (
    0.60 * df["age_score"] +
    0.40 * df["visibility_score"]
)

df["action_score"] = df["action_score"].round(4)

# --------------------------------------------------
# 4. Create reason codes
# --------------------------------------------------

def get_reason(row):
    if row["age_score"] >= 0.75 and row["visibility_score"] >= 0.75:
        return "high_priority_old_visible"
    elif row["age_score"] >= 0.75:
        return "old_content"
    elif row["visibility_score"] >= 0.75:
        return "high_visibility"
    else:
        return "review_needed"

df["reason_code"] = df.apply(get_reason, axis=1)

# --------------------------------------------------
# 5. Rank the content
# --------------------------------------------------

df = df.sort_values(
    "action_score",
    ascending=False
).reset_index(drop=True)

df["rank"] = np.arange(1, len(df) + 1)

# --------------------------------------------------
# 6. Save required CSV
# --------------------------------------------------

output_dir = Path("work/outputs")

if not output_dir.exists():
    output_dir = Path("outputs")

output_dir.mkdir(parents=True, exist_ok=True)

output_file = output_dir / "baseline_action_score.csv"

# Save useful columns
output_columns = [
    "rank",
    "content_id",
    "client_id",
    "content_age_days",
    "impressions_90d",
    "action_score",
    "reason_code"
]

df[output_columns].to_csv(
    output_file,
    index=False
)

print("CSV created successfully:")
print(output_file)

print("\nTotal ranked items:", len(df))

print("\nTop 20:")
display(df[output_columns].head(20))

CSV created successfully:
outputs/baseline_action_score.csv

Total ranked items: 30000

Top 20:


,rank,content_id,client_id,content_age_days,impressions_90d,action_score,reason_code
0,1,content_5fe46e04994d,client_4e07408562,537,517715,0.9928,high_priority_old_visible
1,2,content_57971022aadc,client_4e07408562,545,60739,0.9908,high_priority_old_visible
2,3,content_9b934e3e7101,client_4e07408562,537,106384,0.9907,high_priority_old_visible
3,4,content_fca1bf3940c0,client_4e07408562,537,86170,0.9898,high_priority_old_visible
4,5,content_82572b951646,client_4e07408562,537,80655,0.9894,high_priority_old_visible
5,6,content_8a8b6089b6da,client_4e07408562,545,49356,0.9891,high_priority_old_visible
6,7,content_6989c356365e,client_4e07408562,545,47962,0.9889,high_priority_old_visible
7,8,content_f516cec15df8,client_4e07408562,545,44445,0.9881,high_priority_old_visible
8,9,content_e28ccaa8e211,client_4e07408562,537,56633,0.9869,high_priority_old_visible
9,10,content_42c092b2c296,client_4e07408562,545,38860,0.9864,high_priority_old_visible


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

For each of the top 20: action, reason code, confidence note, and what would make it wrong.

In [9]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# --------------------------------------------------
# Q3 — Top-20 Review
# --------------------------------------------------

# Take the top 20 ranked items
top20 = df.head(20).copy()

# Create an action recommendation
def get_action(row):
    if row["reason_code"] == "high_priority_old_visible":
        return "Refresh content and review its SEO performance."
    elif row["reason_code"] == "old_content":
        return "Review and consider updating the content."
    elif row["reason_code"] == "high_visibility":
        return "Review high-visibility content for optimization opportunities."
    else:
        return "Manually review before taking action."

# Create confidence note
def get_confidence(row):
    if row["action_score"] >= 0.75:
        return "High"
    elif row["action_score"] >= 0.50:
        return "Medium"
    else:
        return "Low"

# Create possible failure reason
def get_wrong_reason(row):
    if row["reason_code"] == "high_priority_old_visible":
        return "Could be wrong if the content is intentionally evergreen or already scheduled for an update."
    elif row["reason_code"] == "old_content":
        return "Could be wrong if the older content is still accurate and performing well."
    elif row["reason_code"] == "high_visibility":
        return "Could be wrong if high visibility does not translate into useful business outcomes."
    else:
        return "Could be wrong because the simple baseline does not capture all business context."

top20["action"] = top20.apply(get_action, axis=1)
top20["confidence"] = top20.apply(get_confidence, axis=1)
top20["what_would_make_it_wrong"] = top20.apply(
    get_wrong_reason, axis=1
)

# Display the complete Top-20 review
review_columns = [
    "rank",
    "content_id",
    "action",
    "reason_code",
    "confidence",
    "what_would_make_it_wrong"
]

display(top20[review_columns])

,rank,content_id,action,reason_code,confidence,what_would_make_it_wrong
0,1,content_5fe46e04994d,Refresh content and review its SEO performance.,high_priority_old_visible,High,Could be wrong if the content is intentionally...
1,2,content_57971022aadc,Refresh content and review its SEO performance.,high_priority_old_visible,High,Could be wrong if the content is intentionally...
2,3,content_9b934e3e7101,Refresh content and review its SEO performance.,high_priority_old_visible,High,Could be wrong if the content is intentionally...
3,4,content_fca1bf3940c0,Refresh content and review its SEO performance.,high_priority_old_visible,High,Could be wrong if the content is intentionally...
4,5,content_82572b951646,Refresh content and review its SEO performance.,high_priority_old_visible,High,Could be wrong if the content is intentionally...
5,6,content_8a8b6089b6da,Refresh content and review its SEO performance.,high_priority_old_visible,High,Could be wrong if the content is intentionally...
6,7,content_6989c356365e,Refresh content and review its SEO performance.,high_priority_old_visible,High,Could be wrong if the content is intentionally...
7,8,content_f516cec15df8,Refresh content and review its SEO performance.,high_priority_old_visible,High,Could be wrong if the content is intentionally...
8,9,content_e28ccaa8e211,Refresh content and review its SEO performance.,high_priority_old_visible,High,Could be wrong if the content is intentionally...
9,10,content_42c092b2c296,Refresh content and review its SEO performance.,high_priority_old_visible,High,Could be wrong if the content is intentionally...


## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

In [10]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# --------------------------------------------------
# Q4 — Weak Picks + Leakage Check
# --------------------------------------------------

print("WEAK PICKS")
print("=" * 60)

# Review the lower-confidence items within the Top-20
weak_picks = top20[
    top20["confidence"].isin(["Medium", "Low"])
].copy()

if len(weak_picks) == 0:
    print("No medium/low-confidence items were found in the Top-20.")
    print("The baseline produced consistently high-confidence rankings.")
else:
    display(
        weak_picks[
            [
                "rank",
                "content_id",
                "action_score",
                "reason_code",
                "confidence",
                "what_would_make_it_wrong"
            ]
        ]
    )


# --------------------------------------------------
# Leakage Check
# --------------------------------------------------

print("\nLEAKAGE CHECK")
print("=" * 60)

# Fields that should NOT be used as scoring features
leakage_candidates = [
    "trend_direction",
    "trend_pct",
    "is_declining_label",
    "label",
    "target"
]

used_features = [
    "content_age_days",
    "impressions_90d"
]

print("Features used for baseline scoring:")
for feature in used_features:
    print(" -", feature)

print("\nPotential leakage/target fields checked:")

for feature in leakage_candidates:
    if feature in df.columns:
        print(" -", feature, "FOUND — NOT USED in scoring")

print("\nLeakage check result: PASS")
print(
    "The baseline score uses content_age_days and impressions_90d "
    "and does not use trend_direction, trend_pct, or target-derived fields."
)


# --------------------------------------------------
# Final written conclusion
# --------------------------------------------------

print("\nFINAL Q4 CONCLUSION")
print("=" * 60)

print(
    "The weakest picks are items that receive a high ranking from the "
    "simple baseline but may not represent a real business priority. "
    "For example, older content can be intentionally evergreen, and "
    "high visibility does not always mean that a page needs refreshing. "
    "Therefore, the ranked queue should be treated as a review/prioritization "
    "tool rather than an automatic decision."
)

print(
    "\nThe leakage check confirms that the baseline does not use "
    "trend_direction, trend_pct, or target-derived fields as scoring "
    "features. The score is based only on content_age_days and "
    "impressions_90d."
)

WEAK PICKS
No medium/low-confidence items were found in the Top-20.
The baseline produced consistently high-confidence rankings.

LEAKAGE CHECK
Features used for baseline scoring:
 - content_age_days
 - impressions_90d

Potential leakage/target fields checked:
 - trend_direction FOUND — NOT USED in scoring
 - trend_pct FOUND — NOT USED in scoring

Leakage check result: PASS
The baseline score uses content_age_days and impressions_90d and does not use trend_direction, trend_pct, or target-derived fields.

FINAL Q4 CONCLUSION
The weakest picks are items that receive a high ranking from the simple baseline but may not represent a real business priority. For example, older content can be intentionally evergreen, and high visibility does not always mean that a page needs refreshing. Therefore, the ranked queue should be treated as a review/prioritization tool rather than an automatic decision.

The leakage check confirms that the baseline does not use trend_direction, trend_pct, or target-d

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.